# 实验 01 · 从训练主循环读懂 slime 架构

slime 将 Megatron 训练与 SGLang rollout 连接起来，允许自定义生成和奖励。Ray 负责进程与远程调用。数据缓冲与传输层连接 prompt、生成样本和训练批次，具体实现随版本演进。

本系列固定官方 commit `79aa41a884ab0bf65222c94418f6357d3c31192b`，不把 main 的后续变化混入课程。先按 README 克隆源码，并设置 SLIME_SOURCE（仅自定义目录需要）。

## 1 · 核对源码，而不是只看框架名称

下格记录关键文件哈希，并检查 train 函数的参数。AST 只解析源码，不导入训练入口、不启动 Ray。


In [ ]:
import importlib.util,json,platform
from pathlib import Path
import course
root=course.source()
report=course.source_report()
print('固定版本:',report['revision'])
print(course.function_signature(root/'train.py','train'))
print('Python:',platform.python_version())
for package in ['torch','ray','sglang']:
    print(package,importlib.util.find_spec(package) is not None)
course.save_report('01_source.json',report)


## 2 · 谁负责什么

| 组件 | 主要职责 | 首先检查的故障 |
|---|---|---|
| Megatron 训练端 | log-prob、梯度、优化器与训练状态 | loss mask、并行配置、梯度与 checkpoint |
| SGLang 生成端 | prompt prefill、采样与生成返回值 | 模板、EOS、截断、采样参数 |
| Ray 与资源编排 | 调度 actor、资源与远程调用 | 资源分配、worker 退出、主机 RAM |
| 数据路径 | prompt、样本分组、buffer、训练批次 | 字段对齐、重复/丢样、数据进度 |
| 权重同步 | 将更新后的 actor 参数交给生成端 | 参数版本、名称映射、是否同步完成 |

已有 verl 经验可以帮助理解职责，但不能直接复制 Hydra 参数或 DataProto 字段。slime 的自定义奖励接收 args 和 Sample；完整 rollout 接口与逐样本生成接口也不同。

## 3 · 阅读实际调用顺序

下面只显示训练循环相关行，省略的上下文仍应在官方 train.py 中阅读。先找到首次权重同步，再找 generate、async_train 和下一次权重同步。

“训练完成”是运行时事件；数据进度与模型/优化器是否共同持久化，还要检查 checkpoint 与恢复协议。异步 producer 可能跨越更新边界，不能简单把提交顺序当作每条样本的实际权重版本。


In [ ]:
text=(root/'train.py').read_text(encoding='utf-8')
markers=['update_weights','training_ready','generate.remote','async_train','training_completed','save_checkpoint','pause_rollout_admission','resume_rollout_admission']
for i,line in enumerate(text.splitlines(),1):
    if any(key in line for key in markers):print(f'{i:4d} {line.strip()}')
assert all(key in text for key in ['actor_model.update_weights()','rollout_manager.generate.remote','actor_model.async_train'])
print('PASS: 源码中存在预期入口；未运行训练。')


## 4 · 环境选择与验收

本次核对的官方快速入门推荐带匹配补丁的 Docker 镜像，GPU CI 主要覆盖 H100/H200，并列出 Blackwell 镜像。当前本机 RTX 4060 8GB、verl 环境没有 SGLang；导入 Sample 成功不能证明 Megatron/SGLang 能共同运行。

完整训练请新建环境，固定源码 commit、镜像 digest、驱动和模型 revision。不要把 `latest` 镜像标签当作可复现版本，也不要在原 verl venv 内直接升级整套 CUDA/推理依赖。

验收：画出一条样本的 prompt → rollout → reward → train → weight sync 路径。说明模型/优化器 checkpoint、rollout 缓冲和 Ray 进程各自保存什么，列出三个无法仅凭退出码判断的错误。


---

<a href="https://github.com/SergioTermann"><img src="assets/logo.png" alt="SergioTermann" width="44" height="44" align="left" style="margin: 0 12px 0 0;"></a>

**更多有趣的 AI 课程**  
[← 访问 SergioTermann 的 GitHub 主页](https://github.com/SergioTermann)